In [ ]:
# IMPORTANT! Set this to where you want to store your copy of the data!
import os
os.environ['ERA5LOWRESDEMO'] = '/scratch/kd24/tjl548/'

import hydra
import pathlib
import xarray as xr

from omegaconf import OmegaConf

import pyearthtools.data.archive
import pyearthtools.tutorial
import pyearthtools.training
import pyearthtools.pipeline

import fourcastnext


In [2]:
# era5_lowres = xr.open_zarr('gs://weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr')

In [3]:
# subset = era5_lowres[['10m_u_component_of_wind', '10m_v_component_of_wind', '2m_temperature', 'mean_sea_level_pressure']]

In [ ]:
# IMPORTANT! Put this somewhere sensible, then set the environment variable as indicated at the start of the tutorial
# subset.to_netcdf('/scratch/kd24/tjl548/era5_lowres.nc')

In [7]:
era5_loaded = xr.open_dataset('/scratch/kd24/tjl548/era5_lowres.nc')
# era5_loaded  # Uncomment this if you want to 

In [6]:
accessor = pyearthtools.data.archive.era5_demo_subset('foo')
accessor['2010-01-01']

<xarray.Dataset> Size: 132kB
Dimensions:                  (time: 4, longitude: 64, latitude: 32)
Coordinates:
  * latitude                 (latitude) float64 256B -87.19 -81.56 ... 87.19
  * longitude                (longitude) float64 512B 0.0 5.625 ... 348.8 354.4
  * time                     (time) datetime64[ns] 32B 2010-01-01 ... 2010-01...
Data variables:
    10m_u_component_of_wind  (time, longitude, latitude) float32 33kB dask.array<chunksize=(4, 36, 18), meta=np.ndarray>
    10m_v_component_of_wind  (time, longitude, latitude) float32 33kB dask.array<chunksize=(4, 36, 18), meta=np.ndarray>
    2m_temperature           (time, longitude, latitude) float32 33kB dask.array<chunksize=(4, 36, 18), meta=np.ndarray>
    mean_sea_level_pressure  (time, longitude, latitude) float32 33kB dask.array<chunksize=(4, 36, 18), meta=np.ndarray>

In [8]:
config_dir = str((pathlib.Path(fourcastnext.__file__).parent / '../../Training/limited_variables_early_stopping').resolve())
                 

In [9]:
initialised = hydra.initialize_config_dir(version_base=None, config_dir = config_dir)
cfg = hydra.compose(config_name="lowres.yaml")

In [10]:
splits = {
    "train_split": pyearthtools.pipeline.iterators.DateRange(*cfg.data.splits.train),
    "valid_split": pyearthtools.pipeline.iterators.DateRange(*cfg.data.splits.valid),
}

In [11]:
pipeline_path = str((pathlib.Path(fourcastnext.__file__).parent / '../../Training/pipelines/low_res_demo_subset.pipe').resolve())

In [12]:
datamodule = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    pipeline_path,
    **splits,
    **cfg.data.module
)

In [13]:
model = hydra.utils.instantiate(cfg.model)

In [14]:
trainer = pyearthtools.training.lightning.Train(
    model,
    datamodule,
    path=cfg.path,
    trainer_kwargs={'num_sanity_val_steps': 0},
    **OmegaConf.to_object(cfg.trainer)
)

In [ ]:
trainer.fit()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/conda/envs/pet/lib/python3.11/site-packages/lightning/pytorch/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type    | Params | Mode 
---------------------------------------------
0 | model    | AFNONet | 60.6 M | train
1 | loss_obj | L1Loss  | 0      | train
---------------------------------------------
60.6 M    Trainable params
0         Non-trainable params
60.6 M    Total params
242.261   Total estimated model params size (MB)
104       Modules in train mode
0         Modules in eval mode
/opt/conda/envs/pet/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` ar

Training: |          | 0/? [00:00<?, ?it/s]